In [1]:
import sys
sys.path.insert(0,'..')
%load_ext autoreload
%autosave 180

Autosaving every 180 seconds


In [2]:
import torch
import pandas as pd
import numpy as np
from scipy.stats import spearmanr
from source.version1.data import dataLoader
from source.version1.model import Model
from source.version1.score import scoreModel
from apex import amp

In [3]:
def evaluateModel(labels, scores):
    correls = []
    for idx in range(30):
        label = labels.iloc[:,idx]
        score = scores.iloc[:,idx]
        correl = spearmanr(label, score).correlation
        correls.append(correl)
    metric = round(np.mean(correls), 4)
    print('Metric:', metric)
    return None

In [4]:
def score(fold):
    loader = {}
    loader['batch_size'] = 2
    loader['fold'] = fold
    _, valid = dataLoader(**loader)
    model = Model()
    weights = torch.load('../../model/version-1/fold-{}/model.pt'.format(fold), map_location='cpu')
    print('Loss:', weights['loss'])
    weights = weights['model_state_dict']
    model.load_state_dict(weights)
    model = model.to('cuda:0')
    model = amp.initialize(model, opt_level="O2", keep_batchnorm_fp32=True, verbosity=0)
    trainer = {}
    trainer['model'] = model
    trainer['data'] = valid
    trainer['save'] = '../../model/version-1/fold-{}/'.format(fold)
    scoreModel(**trainer)
    model = model.cpu()
    del model
    return None

In [5]:
score(1) 

Loss: -0.3931


In [6]:
score(2)

Loss: -0.3834


In [7]:
score(3)

Loss: -0.4005


In [8]:
score(4)

Loss: -0.3864


In [9]:
score(5)

Loss: -0.3841


In [10]:
score_1 = pd.read_csv('../../model/version-1/fold-1/scores.csv', header=None)
score_2 = pd.read_csv('../../model/version-1/fold-2/scores.csv', header=None)
score_3 = pd.read_csv('../../model/version-1/fold-3/scores.csv', header=None)
score_4 = pd.read_csv('../../model/version-1/fold-4/scores.csv', header=None)
score_5 = pd.read_csv('../../model/version-1/fold-5/scores.csv', header=None)
scores = score_1.append(score_2).append(score_3).append(score_4).append(score_5)

In [11]:
label_1 = pd.read_csv('../../model/version-1/fold-1/labels.csv', header=None)
label_2 = pd.read_csv('../../model/version-1/fold-2/labels.csv', header=None)
label_3 = pd.read_csv('../../model/version-1/fold-3/labels.csv', header=None)
label_4 = pd.read_csv('../../model/version-1/fold-4/labels.csv', header=None)
label_5 = pd.read_csv('../../model/version-1/fold-5/labels.csv', header=None)
labels = label_1.append(label_2).append(label_3).append(label_4).append(label_5)

In [12]:
scores.shape, labels.shape

((6078, 30), (6078, 30))

In [13]:
evaluateModel(labels, scores)

Metric: 0.3862


In [14]:
scores.to_csv('../../model/version-1/scores.csv', index=False)
labels.to_csv('../../model/version-1/labels.csv', index=False)